# Creating a Representative Sample Dataset

The original SAML-D dataset contains approximately 9.5 million financial transactions and is highly imbalanced, with suspicious transactions representing only about 0.1% of all records.

To create a computationally manageable dataset suitable for machine learning, all suspicious transactions are retained while a random sample of normal transactions is selected. The resulting dataset preserves the characteristics of the original data while providing a more balanced class distribution for model training and evaluation.

In [14]:
import pandas as pd



DATASET_PATH = "SAML-D.csv"
NORMAL_SAMPLE_SIZE = 190000
CHUNK_SIZE = 100000



target = pd.read_csv(DATASET_PATH,
                     usecols=["Is_laundering"])

print(target["Is_laundering"].value_counts())



suspicious_transactions = []
normal_transactions = []

Is_laundering
0    9494979
1       9873
Name: count, dtype: int64


In [15]:
# Read the dataset in chunks and collect transactions

for chunk in pd.read_csv(DATASET_PATH, chunksize=CHUNK_SIZE):

    # Keep all suspicious transactions
    suspicious = chunk[chunk["Is_laundering"] == 1]
    suspicious_transactions.append(suspicious)

    # Sample normal transactions from the current chunk
    normal = chunk[chunk["Is_laundering"] == 0]

    sample_size = min(
        len(normal),
        NORMAL_SAMPLE_SIZE // 95
    )

    normal_sample = normal.sample(
        n=sample_size,
        random_state=42
    )

    normal_transactions.append(normal_sample)

In [16]:
# Combine all suspicious transactions
suspicious_df = pd.concat(suspicious_transactions, ignore_index=True)

# Combine sampled normal transactions
normal_df = pd.concat(normal_transactions, ignore_index=True)

In [17]:
# Keep exactly the desired number of normal transactions

normal_df = normal_df.sample(
    n=NORMAL_SAMPLE_SIZE,
    random_state=42
)

In [18]:
# Create the final dataset

aml_sample = pd.concat(
    [normal_df, suspicious_df],
    ignore_index=True
)

# Shuffle the dataset

aml_sample = aml_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [19]:
# Display class distribution

print(aml_sample["Is_laundering"].value_counts())

Is_laundering
0    190000
1      9873
Name: count, dtype: int64


In [20]:
print(aml_sample.shape)

(199873, 12)


In [21]:
# Save the sampled dataset

aml_sample.to_csv(
    "AML_sample.csv",
    index=False
)

print("AML_sample.csv has been created successfully.")

AML_sample.csv has been created successfully.


In [22]:
print("=" * 40)
print("Sample dataset created successfully")
print("=" * 40)

print(f"Original suspicious transactions: {len(suspicious_df):,}")
print(f"Sampled normal transactions: {len(normal_df):,}")
print(f"Final dataset size: {len(aml_sample):,}")

Sample dataset created successfully
Original suspicious transactions: 9,873
Sampled normal transactions: 190,000
Final dataset size: 199,873
